# P2 Ex7 — RepDWNet + Weighted CE + Macro Soft Dice training on Kaggle

This notebook trains the P2 Ex7 ablation based on the B1 RepDWNet Run 5 recipe. It keeps the architecture and training protocol fixed, changes only the loss to baseline-compatible Weighted CE plus macro Soft Dice, and uses `lambda=1.0`. Enable **Internet** and attach the Kaggle PhenoBench dataset before running.

The final cell produces one ZIP containing checkpoints, logs, the resolved configuration, and a source revision manifest.

In [ ]:
# Kaggle runtime dependencies. Kaggle supplies torch/CUDA; do not reinstall torch.
!nvidia-smi
!pip install -q --upgrade 'pytorch-lightning==1.9.5' 'torchmetrics==0.11.4'
!pip install -q 'setuptools<70' tensorboard matplotlib pandas tabulate timm scikit-image tifffile albumentations pyyaml rich


In [ ]:
# ---- User settings ---------------------------------------------------------
# Leave DATASET_DIR empty to auto-detect the attached Kaggle input.
REPO_URL = 'https://github.com/FUAVX/RPD_Reproduce.git'
GIT_REF = 'f/wdice-loss'  # pin to a commit SHA after the branch has been pushed
SCENARIO = 'P2_Ex7_RepDWNet_WDice-Loss'
DATASET_DIR = ''  # e.g. '/kaggle/input/phenobench-dataset/PhenoBench'
NUM_WORKERS = 2

import json
import pathlib
import shutil
import subprocess
import yaml

WORKING_DIR = pathlib.Path('/kaggle/working')
REPO_DIR = WORKING_DIR / 'RPD_P2'
RUN_DIR = WORKING_DIR / 'p2_ex7_repdwnet_wdice'

def run(command, cwd=None):
    print('+', ' '.join(map(str, command)))
    subprocess.run(command, cwd=cwd, check=True)

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
run(['git', 'clone', '--branch', GIT_REF, '--single-branch', REPO_URL, str(REPO_DIR)])
run(['git', 'rev-parse', '--verify', 'HEAD'], cwd=REPO_DIR)

scenario_path = REPO_DIR / 'configs' / 'p2_scenarios' / f'{SCENARIO}.yaml'
required_paths = [scenario_path, REPO_DIR / 'test_wdice.py']
missing = [str(path) for path in required_paths if not path.is_file()]
if missing:
    raise FileNotFoundError('Branch does not contain the committed P2 WDice files: ' + ', '.join(missing))

def is_phenobench_root(path):
    return (
        (path / 'train' / 'images').is_dir()
        and (path / 'train' / 'semantics').is_dir()
        and (path / 'val' / 'images').is_dir()
        and (path / 'val' / 'semantics').is_dir()
    )

if DATASET_DIR:
    dataset_dir = pathlib.Path(DATASET_DIR)
else:
    kaggle_input = pathlib.Path('/kaggle/input')
    candidates = [
        kaggle_input / 'datasets' / 'ndhoang2310' / 'phenobench-dataset' / 'PhenoBench',
        kaggle_input / 'phenobench-dataset' / 'PhenoBench',
        kaggle_input / 'datasets' / 'ndhoang2310' / 'phenobench-dataset',
        kaggle_input / 'phenobench-dataset',
    ]
    if kaggle_input.is_dir():
        attached_inputs = [p for p in kaggle_input.iterdir() if p.is_dir()]
        candidates.extend(p / 'PhenoBench' for p in attached_inputs)
        candidates.extend(attached_inputs)
    dataset_dir = next((p for p in candidates if is_phenobench_root(p)), None)
if dataset_dir is None or not is_phenobench_root(dataset_dir):
    raise FileNotFoundError('Cannot find a complete PhenoBench root in /kaggle/input. Attach the PhenoBench dataset, or set DATASET_DIR to its root containing train/images, train/semantics, val/images, and val/semantics.')

print(f'[OK] source ref: {subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()}')
print(f'[OK] dataset: {dataset_dir}')
print(f'[OK] train images: {len(list((dataset_dir / "train" / "images").glob("*")))}')


In [ ]:
# Resolve the Kaggle-specific configuration without modifying the cloned repository.
RUN_DIR.mkdir(parents=True, exist_ok=True)
with scenario_path.open(encoding='utf-8') as stream:
    cfg = yaml.safe_load(stream)

cfg['data']['path_to_dataset'] = str(dataset_dir)
cfg['data']['num_workers'] = NUM_WORKERS
active_cfg_path = RUN_DIR / f'active_config_{SCENARIO}.yaml'
with active_cfg_path.open('w', encoding='utf-8') as stream:
    yaml.safe_dump(cfg, stream, sort_keys=False)

assert cfg['train']['loss'] == 'weighted_dice'
assert cfg['loss']['dice']['lambda'] == 1.0
assert cfg['loss']['dice']['epsilon'] == 1e-6
assert cfg['train']['class_weights'] == [1.47, 5.06, 10.02]
print(yaml.safe_dump({
    'experiment': cfg['experiment'],
    'backbone': cfg['backbone'],
    'loss': cfg['loss'],
}, sort_keys=False))


In [ ]:
# Run the focused loss checks before the B1-matched DDP training recipe.
run(['python', 'test_wdice.py'], cwd=REPO_DIR)

TRAIN_LOG_DIR = RUN_DIR / 'training_output'
run([
    'python', 'multi_metric_train.py',
    '--config', str(active_cfg_path),
    '--export_dir', str(TRAIN_LOG_DIR),
    '--devices', '2',
], cwd=REPO_DIR)


In [ ]:
# Package every artifact needed for evaluation and later analysis.
source_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
manifest = {
    'experiment': SCENARIO,
    'source_repository': REPO_URL,
    'requested_ref': GIT_REF,
    'source_commit': source_commit,
    'dataset_dir_at_runtime': str(dataset_dir),
    'loss': cfg['loss'],
    'class_weights': cfg['train']['class_weights'],
}
(RUN_DIR / 'run_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
archive = shutil.make_archive(str(WORKING_DIR / f'{SCENARIO}_artifacts'), 'zip', root_dir=RUN_DIR)
print(f'[DONE] Download this Kaggle output: {archive}')
print(f'[DONE] Source commit: {source_commit}')
